In [ ]:
from pathlib import Path
import os

# if you want to see where the notebook kernel thinks it is
print("CWD:", Path.cwd())

# ✅ force repo root explicitly
PROJECT_ROOT = Path("/Users/dev/Desktop/graphCeylon-API")
print("PROJECT_ROOT:", PROJECT_ROOT)

INPUT_CSV = PROJECT_ROOT / "data/recommender_service/clean/sltda_arrivals_monthly.csv"
print("INPUT_CSV:", INPUT_CSV, "| exists:", INPUT_CSV.exists())



import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthBegin
from statsmodels.tsa.arima.model import ARIMA

# Paths
INPUT_CSV = PROJECT_ROOT / "data/recommender_service/clean/sltda_arrivals_monthly.csv"

OUTPUT_HYBRID_CSV = PROJECT_ROOT / "models/recommender_service/sltda_arrivals_hybrid_arima_sarima.csv"


PROJECT_ROOT: /Users/dev/Desktop


In [2]:
import pandas as pd



df_arrivals = pd.read_csv(INPUT_CSV, parse_dates=["date"])
df_arrivals = df_arrivals.sort_values("date").set_index("date")

print("Loaded:", df_arrivals.shape)
df_arrivals.head()


FileNotFoundError: [Errno 2] No such file or directory: '/Users/dev/Desktop/data/recommender_service/clean/sltda_arrivals_monthly.csv'

In [ ]:
import numpy as np
import math
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error

VAL_MONTHS = 12
series = df_arrivals["total_arrivals"]

train = series.iloc[:-VAL_MONTHS]
val   = series.iloc[-VAL_MONTHS:]

y_true_val = val.values
val_index  = val.index
print("Validation window:", val_index.min().date(), "→", val_index.max().date())


Validation window: 2024-11-01 → 2025-10-01


In [ ]:
best_arima_order = (3, 2, 3)

def get_arima_val_forecast(series: pd.Series, order, val_months: int = 12):
    train = series.iloc[:-val_months]
    val   = series.iloc[-val_months:]

    model = ARIMA(train, order=order)
    fitted = model.fit()
    forecast_vals = fitted.forecast(steps=val_months)

    return forecast_vals.values  # np.ndarray shape (val_months,)

y_pred_arima = get_arima_val_forecast(series, best_arima_order, val_months=VAL_MONTHS)


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting param

In [ ]:
best_sarima_order = (2, 1, 0)
best_sarima_seasonal = (2, 1, 1, 12)

def get_sarima_val_forecast(series: pd.Series,
                            order,
                            seasonal_order,
                            val_months: int = 12):
    train = series.iloc[:-val_months]
    val   = series.iloc[-val_months:]

    model = SARIMAX(
        train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False)
    forecast_vals = fitted.forecast(steps=val_months)

    return forecast_vals.values

y_pred_sarima = get_sarima_val_forecast(
    series,
    best_sarima_order,
    best_sarima_seasonal,
    val_months=VAL_MONTHS,
)


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without

In [ ]:
print("Shapes:", y_true_val.shape, y_pred_arima.shape, y_pred_sarima.shape)



Shapes: (12,) (12,) (12,)


In [ ]:
rmse_arima = 94103.14
rmse_sarima = 16737.75

inv_var_arima  = 1.0 / (rmse_arima ** 2)
inv_var_sarima = 1.0 / (rmse_sarima ** 2)

w_arima  = inv_var_arima / (inv_var_arima + inv_var_sarima)
w_sarima = inv_var_sarima / (inv_var_arima + inv_var_sarima)

print(f"w_ARIMA  = {w_arima:.4f}")
print(f"w_SARIMA = {w_sarima:.4f}")
print("Check sum:", w_arima + w_sarima)


w_ARIMA  = 0.0307
w_SARIMA = 0.9693
Check sum: 1.0


In [ ]:
def evaluate_forecast(y_true, y_pred, label=""):
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f"[{label}] RMSE = {rmse:.2f}  MAE = {mae:.2f}")
    return rmse, mae

# 1) Pure models
evaluate_forecast(y_true_val, y_pred_arima, label="ARIMA (3,2,3)")
evaluate_forecast(y_true_val, y_pred_sarima, label="SARIMA (2,1,0)x(2,1,1,12)")

# 2) Hybrid with error-based weights
y_pred_hybrid_auto = w_arima * y_pred_arima + w_sarima * y_pred_sarima
rmse_h_auto, mae_h_auto = evaluate_forecast(
    y_true_val, y_pred_hybrid_auto, label="Hybrid (error-based weights)"
)

# 3) Hybrid with manual 0.2 / 0.8 weights (optional experiment)
w_arima_manual  = 0.2
w_sarima_manual = 0.8
y_pred_hybrid_manual = w_arima_manual * y_pred_arima + w_sarima_manual * y_pred_sarima
rmse_h_man, mae_h_man = evaluate_forecast(
    y_true_val, y_pred_hybrid_manual, label="Hybrid (0.2 ARIMA, 0.8 SARIMA)"
)


[ARIMA (3,2,3)] RMSE = 114407.17  MAE = 88624.54
[SARIMA (2,1,0)x(2,1,1,12)] RMSE = 20406.87  MAE = 16511.04
[Hybrid (error-based weights)] RMSE = 22001.51  MAE = 17248.12
[Hybrid (0.2 ARIMA, 0.8 SARIMA)] RMSE = 34974.23  MAE = 26349.52


In [ ]:
from pandas.tseries.offsets import MonthBegin

def forecast_arima_full(series: pd.Series, order, steps: int = 12):
    model = ARIMA(series, order=order)
    fitted = model.fit()
    forecast_vals = fitted.forecast(steps=steps)

    last_date = series.index.max()
    start_future = last_date + MonthBegin(1)
    future_index = pd.date_range(start=start_future, periods=steps, freq="MS")

    return pd.Series(forecast_vals.values, index=future_index)

def forecast_sarima_full(series: pd.Series,
                         order,
                         seasonal_order,
                         steps: int = 12):
    model = SARIMAX(
        series,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False)
    forecast_vals = fitted.forecast(steps=steps)

    last_date = series.index.max()
    start_future = last_date + MonthBegin(1)
    future_index = pd.date_range(start=start_future, periods=steps, freq="MS")

    return pd.Series(forecast_vals.values, index=future_index)

# Individual future forecasts
arima_future  = forecast_arima_full(series, best_arima_order, steps=12)
sarima_future = forecast_sarima_full(series, best_sarima_order, best_sarima_seasonal, steps=12)

# Choose weights (e.g. error-based or manual)
wA = w_arima        # or w_arima_manual
wS = w_sarima       # or w_sarima_manual

hybrid_future = wA * arima_future + wS * sarima_future

# Build final DataFrame
hybrid_future_df = pd.DataFrame({
    "date": hybrid_future.index,
    "forecast_hybrid": hybrid_future.values,
    "forecast_arima": arima_future.values,
    "forecast_sarima": sarima_future.values,
})

hybrid_future_df.to_csv(OUTPUT_HYBRID_CSV, index=False)
print(f"Saved hybrid forecast → {OUTPUT_HYBRID_CSV}")



e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting param

Saved hybrid forecast → data_clean/forecast/sltda_arrivals_hybrid_arima_sarima.csv


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
